<a href="https://colab.research.google.com/github/Maverick-Ansh/recurrent-depth-4b/blob/main/Copy_of_recurrent_depth_4b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent Depth: reproducing arXiv:2502.05171, and retrofitting it onto a 4B model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maverick-Ansh/recurrent-depth-4b/blob/main/recurrent_depth_4b.ipynb)

Geiping et al., *"Scaling up Test-Time Compute with Latent Reasoning: A Recurrent
Depth Approach"* trains a 3.5B model that iterates a shared **core block** at
test time, unrolling to arbitrary depth without emitting a single extra token.

    e  = P(x)                     prelude embeds the input
    s0 ~ N(0, sigma^2 I)          random initial latent state
    si = R(e, s_{i-1})            core block, run r times, e re-injected every step
    p  = C(sr)                    coda un-embeds and predicts

This notebook runs two tracks on 2x T4:

* **Track A** -- the architecture from scratch at the paper's own small shape
  `(lP, lR, lC) = (1, 4, 1)`, trained on a **depth-controlled task suite** where
  the sequential depth a problem requires is a knob with exact ground truth.
* **Track B** -- surgery on **Qwen3-4B-Base**: 36 layers cut into
  prelude / looped core / coda, the paper's concat adapter installed, and the
  random-*r* objective used to teach a pretrained model to use a recurrence it
  never had. That retrofit-vs-pretrain question is the paper's own (Sec. 6.3).

Full method and results: [`REPORT.md`](https://github.com/Maverick-Ansh/recurrent-depth-4b/blob/main/REPORT.md).

In [ ]:
!git clone -q https://github.com/Maverick-Ansh/recurrent-depth-4b.git /content/recurrent-depth-4b 2>/dev/null || (cd /content/recurrent-depth-4b && git pull -q)
%cd /content/recurrent-depth-4b
import torch, subprocess
print("torch", torch.__version__, "| gpus", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} sm_{p.major}{p.minor} {p.total_memory/1e9:.1f} GB")
# T4 is sm_75: bf16 is emulated there, so everything below runs fp16 + GradScaler.

/content/recurrent-depth-4b
torch 2.10.0+cu128 | gpus 2
  [0] Tesla T4 sm_75 15.6 GB
  [1] Tesla T4 sm_75 15.6 GB


## 1. The paper's rules, as executable assertions

`scripts/smoke.py` asserts the *rules* of the paper rather than tensor shapes.
The one worth pointing at: we build the paper's own `(2,4,2)`, `h=5280` config on
the meta device and check that our parameter layout reproduces the 3.5B / 1.5B /
0.5B split **and the upper x-axis of the paper's Figure 1** (materialized
parameters at r = 1, 4, 6, 8, 12, 20, 32, 48, 64). If the adapter or the norms
were wired wrongly, that axis would not land.

In [ ]:
!python scripts/smoke.py


== Sec. 3.3 / Fig. 3 : the unrolling distribution Lambda ==
  PASS  Lambda moments match Fig. 3 (mean 33.0, median 29.0 at rbar=32)   [{'mean': 32.983428955078125, 'median': 29.0, 'mode': 22.0}]
  PASS  r >= 1 always (Eq. 2 adds +1 to the Poisson draw)   [min=1 max=61]
  PASS  Lambda is heavy-tailed: P(r > 2*rbar) > 0 (Sec. 3.3 'heavy tail')   [P(r > 2*rbar) = 0.0810]

== Sec. 4.1 : initialisation ==
  PASS  sigma_h^2 = 2/(5h) on ordinary weights   [sigma_h=0.02795 emp=0.02761]
  PASS  sigma_out^2 = 1/(5hl) on out-projections, and smaller than sigma_h   [sigma_out=0.003390 emp=0.003348 (l=34)]
  PASS  var(s0) = 2/5 = var(gamma*E(x)) [Sec. 4.1 consistency]   [var(gamma*E)=0.390  var(s0)=0.389  (paper: both 0.4)]

== Sec. 3.2 : architecture layout ==
  PASS  core block starts with adapter A : R^2h -> R^h (Sec. 3.2)   [A: (512, 1024)]
  PASS  sandwich norm has n1..n4; pre-norm ablation has only n1,n3   [sandwich: n1..n4 ; pre: n1,n3]
  PASS  tied input/output embeddings   [lm_head.weight

## 2. Gate the evaluation before spending any GPU time

On a resized reproduction the *evaluation* breaks more often than the model.
This gate measures, for every (task x difficulty) cell: the grader's ceiling
against gold answers, the floor of the best constant policy, and how much of the
prompt space is small enough to be memorised rather than computed.

In [ ]:
!python scripts/check_eval.py

PHASE-4 EVALUATION GATE  --  bracketing every cell before the sweep

[1] Grader ceiling: an oracle that reads the gold answer must score 1.000
    PASS -- grader recovers gold answers on all cells

[2] Per-cell floor: best constant policy, and a no-information model
    cell               const-guess  zero-model  headroom  status
    perm/2                   0.199       0.000     0.801  ok
    perm/4                   0.086       0.000     0.914  ok
    perm/8                   0.035       0.000     0.966  ok
    perm/16                  0.017       0.000     0.984  ok
    perm/24                  0.013       0.000     0.987  ok
    add/(2, 1)               0.116       0.000     0.884  ok
    add/(3, 1)               0.085       0.000     0.915  ok
    add/(4, 1)               0.070       0.000     0.930  ok
    add/(2, 2)               0.017       0.000     0.984  ok
    add/(3, 2)               0.013       0.000     0.987  ok
    add/(2, 3)               0.004       0.000     0.996  

## 3. Track A -- pretrain the architecture and its ablations

Twelve arms across two GPUs. Each names the claim it exists to test: the
non-recurrent twin of Table 4, a FLOP-matched version of that twin (a control
the paper does not run), input-injection off, pre-norm instead of sandwich norm,
fixed `s0`, full backprop instead of truncated, and a high-learning-rate arm
that tries to reproduce the Sec. 4.3 collapse.

In [ ]:
import subprocess, sys, os
os.makedirs("logs", exist_ok=True)
p = subprocess.Popen([sys.executable, "scripts/run_sweep.py", "--steps", "2500",
                      "--lr", "3e-4", "--gpus", "0,1", "--out", "results"],
                     stdout=open("logs/sweep.log", "w"), stderr=subprocess.STDOUT,
                     env=dict(os.environ, PYTHONUNBUFFERED="1"), start_new_session=True)
print("sweep pid", p.pid, "-- poll logs/sweep.log; do not block this kernel")

sweep pid 367 -- poll logs/sweep.log; do not block this kernel


In [ ]:
# poll in a SHORT cell -- a long-running cell blocks the Colab kernel with no interrupt
print(open("logs/sweep.log").read()[-2000:])

In [ ]:
!python scripts/analyze.py --results results --figures figures

no results yet


## 4. Sections 6 and 7 -- what recurrence gives you for free

Path independence, extrapolation past the training depth, the zero-shot KL
adaptive exit of Sec. 6.1, KV-cache sharing of Sec. 6.2, and the latent-space
trajectories of Sec. 7 (distance-to-limit map and PCA projections).

In [ ]:
!python scripts/analyze_mechanisms.py --ckpt results/rec_s0.pt

Traceback (most recent call last):
  File "/content/recurrent-depth-4b/scripts/analyze_mechanisms.py", line 181, in <module>
    main()
  File "/content/recurrent-depth-4b/scripts/analyze_mechanisms.py", line 87, in main
    m, cfg = load_model(args.ckpt, dev)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/recurrent-depth-4b/scripts/analyze_mechanisms.py", line 41, in load_model
    d = torch.load(ckpt, map_location=device, weights_only=False)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__i

## 5. Track B -- retrofit recurrent depth onto Qwen3-4B-Base

First the gate: with the identity adapter `A = [0 | I]` the core ignores `s`, so
at `r=1` the retrofit must be **exactly** the base model. That is what makes the
later r-curve interpretable -- anything it gains, it gained from recurrence.

It also starts life inside the failure mode Sec. 4.3 describes for their second
failed run: *"the model has learned early to ignore the incoming state s"*. So
`--adapter-init paper` is run as the contrast arm.

In [ ]:
!python scripts/prep_retrofit_data.py --tokens 6000000 --val-tokens 250000

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 2.65MB/s]
tokenizer_config.json: 9.68kB [00:00, 18.0MB/s]
vocab.json: 2.78MB [00:00, 43.5MB/s]
merges.txt: 1.67MB [00:00, 72.7MB/s]
tokenizer.json: 7.03MB [00:00, 85.0MB/s]
README.md: 26.4kB [00:00, 10.8MB/s]
Resolving data files: 100%|██████████████| 2410/2410 [00:00<00:00, 14799.03it/s]
[ok] HuggingFaceFW/fineweb-edu: 2,814,062 tokens
README.md: 4.80kB [00:00, 6.68MB/s]
Resolving data files: 100%|█████████████████| 114/114 [00:00<00:00, 8238.72it/s]
[ok] open-web-math/open-web-math: 2,214,363 tokens
README.md: 3.30kB [00:00, 6.85MB/s]
[skip] bigcode/the-stack-smol: DatasetNotFoundError: Dataset 'bigcode/the-stack-smol' is a gated dataset on the Hub. You must be authenticated to access it.
train 4,777,840 tokens, val 250,000 tokens -> data_cache
per-source: {'HuggingFaceFW/fineweb-edu': 2814062, 'open-web-math/open-web-math': 2214363, 'bigcode/the-stack-smol': 0}


In [ ]:
!python scripts/verify_retrofit.py --split 9,18,9 --seq 256 --batch 2 --k 2

building retrofit  split=(9, 18, 9)  from Qwen/Qwen3-4B-Base
model.safetensors.index.json: 32.8kB [00:00, 45.5MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:42<00:00, 14.14s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:42<00:00, 189MB/s]
Loading weights: 100%|█| 398/398 [00:16<00:00, 23.51it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 488kB/s]
Traceback (most recent call last):
  File "/content/recurrent-depth-4b/scripts/verify_retrofit.py", line 202, in <module>
    main()
  File "/content/recurrent-depth-4b/scripts/verify_retrofit.py", line 53, in main
    m, tok = build_retrofit(args.model, split=split, adapter_init="identity",
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/recurrent-depth-4b/recurrent_depth/retrofit.py", line 353, in build_retrofit
    hf.to(device).eval()
    ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tran

In [ ]:
import subprocess, sys, os
ENV = dict(os.environ, PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
jobs = [("0", ["--adapter-init", "identity", "--tag", "retro_identity"]),
        ("1", ["--adapter-init", "paper",    "--tag", "retro_paper"])]
for gpu, extra in jobs:
    subprocess.Popen([sys.executable, "scripts/train_retrofit.py", "--steps", "600",
                      "--rbar", "4", "--k", "2", "--lr", "1e-4"] + extra,
                     stdout=open(f"logs/{extra[-1]}.log", "w"), stderr=subprocess.STDOUT,
                     env=dict(ENV, CUDA_VISIBLE_DEVICES=gpu), start_new_session=True)
print("launched both retrofit arms")

launched both retrofit arms


In [ ]:
for t in ["retro_identity", "retro_paper"]:
    print(f"===== {t} =====")
    print("".join(l for l in open(f"logs/{t}.log") if l.startswith("["))[-1500:])

===== retro_identity =====

===== retro_paper =====



In [ ]:
!python scripts/eval_benchmarks.py --base-only --n 300 --out results/bench_base.json
!python scripts/eval_benchmarks.py --ckpt results/retro_identity_trainable.pt \
    --r 1,2,4,8,16 --n 300 --out results/bench_retro_identity.json

Loading weights: 100%|█| 398/398 [00:43<00:00,  9.08it/s, Materializing param=mo
Traceback (most recent call last):
  File "/content/recurrent-depth-4b/scripts/eval_benchmarks.py", line 154, in <module>
    main()
  File "/content/recurrent-depth-4b/scripts/eval_benchmarks.py", line 115, in main
    m, tok = build_retrofit(args.model, split=split, adapter_init=args.adapter_init, device=dev)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/recurrent-depth-4b/recurrent_depth/retrofit.py", line 353, in build_retrofit
    hf.to(device).eval()
    ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 3587, in to
    return super().to(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1381, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist

## 6. Results

Every claim, its verdict, what broke in the evaluation before it worked, and what
was **not** tested, are written up in [`REPORT.md`](https://github.com/Maverick-Ansh/recurrent-depth-4b/blob/main/REPORT.md).

The one thing to carry away if you read nothing else: on a resized reproduction,
budget for the evaluation breaking before the model does. Three of our
measurements were wrong before any of them were right -- a val loss averaged over
irreducibly-random prompt bytes that was flat while accuracy climbed, a task
whose easy cells were memorisable rather than computable, and a KV cache whose
prefill double-counted its own tokens.